# Intervals and real-photo swap on the diffusion benchmarks

Reads the per-crop scores that `score_diffusion_benchmarks.ipynb` uploads (`HF_MODEL/scores/<test set>.npz`) and, without a GPU:

1. gives every detector's AUC and BPCER at APCER ≤ 5% on DiFF and DiffusionFace a 95% interval from a bootstrap over whole subjects (`bootstrap_interval`; DiFF fakes share 143 identities, so resampling crops alone would understate the uncertainty);
2. gives the paired difference between v2 and the official GenD PE on DiFF, from the same resamples;
3. swaps the real side while keeping the fakes fixed: each benchmark's fakes against its own reals and against the other's. If DiffusionFace's low AUC comes from its CelebA-HQ reals, DiFF's fakes against CelebA-HQ reals fall the same way, and DiffusionFace's fakes against DiFF's reals rise.

The swap also changes resolution and framing of the reals (DiffusionFace images are 256 px; DiFF's are larger), so a change it shows belongs to the real photos as a whole, not to CelebA-HQ's processing alone.

BPCER at APCER ≤ 5% sets the threshold on the scored crops themselves, the usual reporting point; a threshold fixed beforehand would give a higher BPCER.

**Runtime:** any Colab runtime, no GPU. **Needs** the Colab secret `HF_TOKEN`.

### Configuration

In [ ]:
REPO_SOURCE = "git+https://github.com/paulohenriquevn/face-decision-lab@workspace"
HF_MODEL = "paulohenriquevn/face-decision-clip-ln"
TEST_SETS = ("cross_diff", "cross_diffusionface")
DETECTORS = ("gend_pe", "gend_ensemble_clip_pe", "v2_ffpp_real", "fusion_max_v2_ffpp_real_gendens")
ROUNDS = 1000
REPORT_PATH = "analysis/diffusion_benchmarks.json"

In [ ]:
%%capture
!pip install -q "face-decision-lab @ {REPO_SOURCE}" huggingface_hub

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token = userdata.get("HF_TOKEN"))

### Scores

In [ ]:
from huggingface_hub import hf_hub_download
from face_decision_lab.score_files import load_scores

loaded = {name: load_scores(hf_hub_download(HF_MODEL, f"scores/{name}.npz")) for name in TEST_SETS}
for name, (samples, scores) in loaded.items():
    print(name, len(samples), "crops,", sum(s.is_bona_fide for s in samples), "bona fide,", sorted(scores))

### 1. Intervals

In [ ]:
import json
import numpy as np
from face_decision_lab.metrics import auc, bpcer_at_apcer
from face_decision_lab.protocol import MAX_APCER, bootstrap_interval

def auc_of(part, values):
    return auc(values, [s.attack_type for s in part])

def bpcer_of(part, values):
    return bpcer_at_apcer(values, [s.attack_type for s in part], MAX_APCER).bpcer

report = {"intervals": {}, "paired_difference": {}, "real_swap": {}}
for name, (samples, scores) in loaded.items():
    for detector in DETECTORS:
        values = scores[detector].tolist()
        row = {"auc": auc_of(samples, values), "auc_95": bootstrap_interval(samples, values, auc_of, ROUNDS),
               "bpcer": bpcer_of(samples, values), "bpcer_95": bootstrap_interval(samples, values, bpcer_of, ROUNDS)}
        report["intervals"][f"{name}/{detector}"] = row
        print(f"{name:20s} {detector:34s} AUC {row['auc']:.3f} [{row['auc_95'][0]:.3f}, {row['auc_95'][1]:.3f}]"
              f"  BPCER {row['bpcer']:.1%} [{row['bpcer_95'][0]:.1%}, {row['bpcer_95'][1]:.1%}]")

### 2. Paired difference on DiFF: v2 minus GenD PE

In [ ]:
samples, scores = loaded["cross_diff"]
pair = np.stack([scores["v2_ffpp_real"], scores["gend_pe"]], axis = 1)
indices = list(range(len(samples)))

def difference(metric):
    def statistic(part, chosen):
        return metric(part, pair[chosen, 0].tolist()) - metric(part, pair[chosen, 1].tolist())
    return statistic

for label, metric in (("auc", auc_of), ("bpcer", bpcer_of)):
    point = difference(metric)(samples, indices)
    low, high = bootstrap_interval(samples, indices, difference(metric), ROUNDS)
    report["paired_difference"][label] = {"point": point, "95": [low, high]}
    print(f"v2 - GenD PE, {label}: {point:+.4f} [{low:+.4f}, {high:+.4f}]")

### 3. Real-photo swap with fixed fakes

In [ ]:
fakes = {name: [i for i, s in enumerate(samples) if not s.is_bona_fide] for name, (samples, _) in loaded.items()}
reals = {name: [i for i, s in enumerate(samples) if s.is_bona_fide] for name, (samples, _) in loaded.items()}
for fake_set in TEST_SETS:
    for real_set in TEST_SETS:
        part = [loaded[fake_set][0][i] for i in fakes[fake_set]] + [loaded[real_set][0][i] for i in reals[real_set]]
        for detector in DETECTORS:
            values = (loaded[fake_set][1][detector][fakes[fake_set]].tolist()
                      + loaded[real_set][1][detector][reals[real_set]].tolist())
            point, interval = auc_of(part, values), bootstrap_interval(part, values, auc_of, ROUNDS)
            report["real_swap"][f"{fake_set} fakes/{real_set} reals/{detector}"] = {"auc": point, "auc_95": interval}
            print(f"{fake_set:20s} fakes vs {real_set:20s} reals  {detector:34s} AUC {point:.3f} [{interval[0]:.3f}, {interval[1]:.3f}]")

### Upload

In [ ]:
from huggingface_hub import HfApi

HfApi().upload_file(path_or_fileobj = json.dumps(report, indent = 2).encode(), repo_id = HF_MODEL,
                    path_in_repo = REPORT_PATH, commit_message = "Intervals and real-photo swap on DiFF and DiffusionFace")